# SelfGPT - Fine-tuning Pipeline (Unsloth QLoRA)

Run this notebook on **Kaggle** or **Google Colab** to fine-tune open-source models using the data exported from your SelfGPT instance.
It uses Unsloth for 2x faster training and less memory usage.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Or mistral/qwen
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## Load Dataset
Upload the `selfgpt_dataset.jsonl` exported from your SelfGPT admin panel, or load it directly from an API endpoint.

In [ ]:
# Upload your dataset to Kaggle/Colab first, then load it:
dataset = load_dataset("json", data_files="selfgpt_dataset.jsonl", split="train")

print(f"Loaded {len(dataset)} training examples.")

chat_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{ai_output}<|eot_id|>"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    systems = examples["system"]
    users = examples["user"]
    assistants = examples["assistant"]
    texts = []
    for s, u, a in zip(systems, users, assistants):
        text = chat_template.format(system_prompt=s, user_input=u, ai_output=a) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Save the LoRA adapters
model.save_pretrained("selfgpt_lora_model") # Local saving
tokenizer.save_pretrained("selfgpt_lora_model")

# You can also push to HuggingFace or export to GGUF for Ollama
# model.push_to_hub("your_name/selfgpt_lora_model", token = "...")

# Export to Ollama / GGUF (q4_k_m)
if False:
    model.save_pretrained_gguf("selfgpt_gguf", tokenizer, quantization_method = "q4_k_m")